# Modeling analyses

**Workflow at a glance**

**Imports** — load every module the notebook uses.

1. **Extract and save modeling input data** — convert per-session loader output into the per-pipeline modeling-input pickle and run the predictor-diagnostics audits (`_collinearity.pkl` + `_timescales.pkl`).
2. **Predictor diagnostics plots** — visualise the audit artifacts before committing to model fitting.
3. **Consolidate the cluster outputs** — merge the per-feature univariate and per-step model-selection pickles (produced on the cluster by `main_univariate_dispatcher` / `main_model_selection_dispatcher`) into one artifact per analysis.
4. **Univariate-model visualisations** — feature ranking / filters for the scalar single-target analyses (vocal onsets, bout parameters, binomial categories) and the per-class multinomial univariate heatmap.
5. **Model-selection visualisations** — forward-stepwise trajectories for the scalar single-target analyses and the per-class multinomial selection (trajectory / filters / diagnosis).
6. **Continuous-manifold visualisations** — selection trajectory and final temporal filters for the continuous acoustic-manifold target.
7. **CNN deep-learning pipeline** — a non-linear model for the continuous manifold-position target.

Each section defines its own parameters in a small cell at its top (paths are written `/mnt/falkner/...` and `configure_path()`-normalised to the host OS); nothing downstream redefines them.

**Settings file**: every pipeline reads `_parameter_settings/modeling_settings.json` via `modeling_settings_dict=None` (loads the project default). To run a one-off override, pass an explicit dict.

In [ ]:
### Imports

import pathlib

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style

from usv_playpen.modeling.consolidate_univariate_results import consolidate as consolidate_univariate
from usv_playpen.modeling.consolidate_model_selection_results import consolidate as consolidate_model_selection
from usv_playpen.modeling.modeling_usv_manifold_position import ContinuousModelingPipeline
from usv_playpen.modeling.modeling_vocal_bout_parameters import BoutParameterPipeline
from usv_playpen.modeling.modeling_vocal_categories_multinomial import MultinomialModelingPipeline
from usv_playpen.modeling.modeling_vocal_onsets import VocalOnsetModelingPipeline
from usv_playpen.modeling.jax_neural_network_cnn import NeuralContinuousCNNRunner

from usv_playpen.visualizations.modeling_plots import (
    DeepResultsVisualizer,
    plot_collinearity_audit,
    plot_feature_ranking,
    plot_manifold_multivariate_filters,
    plot_manifold_selection_trajectory,
    plot_model_selection_results,
    plot_multinomial_multivariate_filters,
    plot_multinomial_selection_diagnosis,
    plot_multinomial_selection_trajectory,
    plot_raw_feature_difference,
    plot_significant_filters,
    plot_significant_filters_grid,
    plot_timescale_audit,
    plot_timescale_audit_per_feature,
    plot_univariate_multinomial_performance,
)

apply_plot_style()


## 1. Extract and save modeling input data

Each of the four extraction calls below converts session-level loader output into the modeling-input pickle that the downstream univariate / model-selection runners consume. They differ only in **what gets predicted** (and therefore which event timestamps populate the timescale audit's `Y(t)` impulse trace):

| Pipeline | Predicts | `Y(t)` impulses |
|---|---|---|
| `VocalOnsetModelingPipeline` | "does a frame start a vocal event (bout / USV)?" | bout / USV onsets |
| `BoutParameterPipeline` | per-bout duration / complexity / intensity | bout starts |
| `MultinomialModelingPipeline` | per-USV vocal category | per-USV starts |
| `ContinuousModelingPipeline` | per-USV UMAP manifold position | per-USV starts |

**Side effects (every call):**

- Writes the modeling-input pickle to `io.save_directory` (filename embeds the cohort label, e.g. `male_mute_partner`, plus a timestamp).
- Writes the paired predictor-diagnostics audit artifacts: `*_collinearity.pkl` and `*_timescales.pkl` (visualised in the next section).

The cohort label is derived from `io.session_list_file` (see `derive_experimental_condition` in `modeling_metadata.py`).

In [ ]:
### Extract and save vocal onset to model

VocalOnsetModelingPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data()


In [ ]:
### Extract and save vocal bout duration/complexity/intensity data to model

BoutParameterPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data()


In [ ]:
### Extract and save category prediction data for multinomial modeling

MultinomialModelingPipeline(modeling_settings_dict=None).extract_and_save_multinomial_input_data()


In [ ]:
### Extract and save USV UMAP target position data to model

ContinuousModelingPipeline(modeling_settings_dict=None).extract_and_save_continuous_data()


## 2. Predictor diagnostics: collinearity + timescale plots

Run **after** the extract step has produced the audit artifacts. The three plots share feature ordering and per-group colour so that one feature can be cross-referenced by row position and hue across all three figures.

**What each plot answers**

- `plot_timescale_audit_per_feature` — for each predictor: how long does its ACF stay above the circular-shift null (ACF horizon, left panel), and at what lag does its cross-correlation with `Y(t)` cross out of the null envelope (XC horizon, right panel)?
- `plot_timescale_audit` — cohort summary as horizontal bars, sorted by horizon length. The triangle markers on the per-feature plot **are** the bar lengths here.
- `plot_collinearity_audit` — Spearman ρ heatmap (left) and VIF horizontal bars (right). Cells whose `|ρ|` exceeds `outline_threshold` are annotated.

**Inputs**: paths to the `*_timescales.pkl` and `*_collinearity.pkl` artifacts from the previous section. The collinearity path is derived from the timescale path by string replacement.

**Settings dependencies** read from inside the audits:

- `diagnostics.timescale_*` — max lag, n_shuffles, shuffle range, signal floor, min run length.
- `model_params.mixture_model_component_index`.
- `model_params.mixture_model_z_score`.
- `mixture_model_params` — the IBI threshold reference line.


In [ ]:
# Parameters -- predictor diagnostics

diag_timescale_pkl = configure_path('/mnt/falkner/Bartul/modeling/modeling_boutparam_bout_durations_male_mute_partner_20260506_122135_timescales.pkl')
diag_collinearity_pkl = diag_timescale_pkl.replace('_timescales.pkl', '_collinearity.pkl')
diag_save_fig_bool = False
diag_save_fig_format = 'svg'
diag_save_fig_directory = pathlib.Path(configure_path('/mnt/falkner/Bartul/figures'))

In [ ]:
### Predictor diagnostics: collinearity + timescale audit plots

# Run these AFTER the extract-and-save step has produced the paired
# `_collinearity.pkl` and `_timescales.pkl` audit artifacts next to the
# main modeling input pickle. The three plots are intended to be read
# side-by-side: row order, per-feature group colour, and the
# single-legend convention are deliberately aligned across all three.
# Paths / toggles live in the Parameters cell (diag_*).

diag_save_fig_directory.mkdir(exist_ok=True, parents=True)

# Order matters: run the per-feature plot FIRST, then the cohort summary.
# The cohort plot (`plot_timescale_audit`) summarises the per-feature
# horizon markers, so the per-feature plot is the ground-truth view.

# (1) Per-feature small-multiples (ACF + cross-correlation horizons).
plot_timescale_audit_per_feature(diag_timescale_pkl,
                                 save_plot_bool=diag_save_fig_bool,
                                 save_dir=diag_save_fig_directory,
                                 plot_format=diag_save_fig_format)

# (2) Cohort timescale summary: horizontal bars of per-feature horizons.
plot_timescale_audit(diag_timescale_pkl,
                     save_plot_bool=diag_save_fig_bool,
                     save_dir=diag_save_fig_directory,
                     plot_format=diag_save_fig_format)

# (3) Collinearity diagnostic: Spearman rho heatmap (left) + VIF bars (right),
#     sharing the timescale plots' feature ordering and per-group palette.
plot_collinearity_audit(diag_collinearity_pkl,
                        save_plot_bool=diag_save_fig_bool,
                        save_dir=diag_save_fig_directory,
                        plot_format=diag_save_fig_format)


## 3. Consolidate per-feature univariate + per-step model-selection pickles

Run this **after** the SLURM job array (`main_univariate_dispatcher.py` / `main_model_selection_dispatcher.py`) finishes. The consolidators:

- assert metadata equality across every per-feature / per-step pickle (guards against stray files from a different run);
- hoist the agreed-upon `_input_metadata` / `_run_metadata` / `_univariate_metadata` blocks to the top of the consolidated artifact;
- emit a self-describing filename built from those blocks.

Use `delete_individuals_after=True` once you have verified the consolidated artifact is correct (the per-feature files become redundant and bloat the directory). `move_to_steps_subdir=True` on the model-selection consolidator relocates the consumed step pickles into `<input_dir>/steps/` so the working directory stays tidy.


In [ ]:
# Parameters -- consolidate the cluster's per-feature / per-step pickles

cons_univariate_input_dir = configure_path('/mnt/falkner/Bartul/modeling/cluster/univariate_results_multi_file/male/male_multinomial_vae_supercategory')
cons_univariate_delete_individuals_after = False
cons_selection_input_dir = configure_path('/mnt/falkner/Bartul/modeling/model_selection_results/male/vocal_onset')
cons_selection_move_to_steps_subdir = False
cons_selection_ignore_provenance_keys = ('git_commit', 'git_dirty', 'package_version', 'settings_sha256')

In [ ]:
### Consolidate per-feature univariate pickles + per-step model-selection pickles

# Run this *after* the SLURM job array finishes. The consolidators assert
# metadata equality across every per-feature / per-step pickle, hoist the
# agreed `_input_metadata` / `_run_metadata` / `_univariate_metadata` blocks
# to the top of the consolidated artifact, and emit a self-describing
# filename. Set `cons_univariate_delete_individuals_after=True` (Parameters
# cell) once you have verified the consolidated artifact is correct.

# 1. Univariate per-feature -> consolidated `univariate_<tag>_<condition>_<ts>.pkl`

consolidate_univariate(
    input_dir=cons_univariate_input_dir,
    delete_individuals_after=cons_univariate_delete_individuals_after,
)


In [ ]:
# 2. Model-selection per-step -> consolidated
#    `selection_<tag>_<condition>_<selection_function>_<ts>.pkl`
# `cons_selection_move_to_steps_subdir=True` relocates consumed step pickles
# into `<input_dir>/steps/`. `cons_selection_ignore_provenance_keys` extends
# the default provenance-key exclusions with `'settings_sha256'` so an
# unrelated mid-run edit to `modeling_settings.json` does not abort the
# `_run_metadata` equality check (drop it to restore the safety net).

consolidate_model_selection(
    input_dir=cons_selection_input_dir,
    move_to_steps_subdir=cons_selection_move_to_steps_subdir,
    ignore_provenance_keys=cons_selection_ignore_provenance_keys,
)

## 4. Univariate-model visualisations

Plots over the consolidated **univariate** pickle (one fit per behavioural feature, no forward stepping). Use these to triage which features look promising before committing to model selection, or to compare cohorts. This section covers **two target families**, separated by the markdown headers below.

**Scalar single-target analyses** — the plotters below serve any analysis that predicts a single scalar target per feature:

- **Vocal onsets** (`VocalOnsetModelingPipeline`) — binary: does a frame start a bout / an individual USV; metric `ll` (log-loss) or `auc`.
- **Bout parameters** (`BoutParameterPipeline`) — regression on per-bout duration / complexity / intensity; metric `explained_deviance` (McFadden's D²) or `spearman_r`.
- **Binomial USV category** (`VocalCategoryModelingPipeline`) — one target category vs a pooled "other" (binary one-vs-rest); metric `ll` or `auc`.

Point the `uni_*` paths at the chosen analysis's consolidated univariate pickle and set the metric accordingly, then run:

- `plot_feature_ranking` — per-feature bar chart of the chosen primary metric, with the secondary metric overlaid. P-value-thresholded features are highlighted.
- `plot_significant_filters` — small multiples of each significant feature's learned filter (one panel per feature).
- `plot_significant_filters_grid` — same filters laid out on a single grid, baseline-corrected for direct cross-feature comparison.
- `plot_raw_feature_difference` — distribution of raw (z-scored) feature values between two conditions, with bootstrapped CIs. Quick sanity check that the feature carries any signal at all.

In [ ]:
# Parameters -- univariate-model visualisations

figures_dir = configure_path('/mnt/falkner/Bartul/modeling/figures')

# Human-readable feature labels are resolved centrally (planned:
# FeatureZoo on main); pass `feature_label_overrides=` to a plotter
# only for a one-off rename.

uni_ranking_results = configure_path('/mnt/falkner/Bartul/modeling/univariate_results/univariate_multinomial_vae_supercategory_intact_partners_male_20260511_204026Z.pkl')
uni_ranking_p_val = 0.01
uni_filters_results = configure_path('/mnt/falkner/Bartul/modeling/gam_results_male_mute_partner_category_18_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl')
uni_filters_grid_results = configure_path('/mnt/falkner/Bartul/modeling/gam_results_male_category_10_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl')
uni_raw_diff_pkl = configure_path('/mnt/falkner/Bartul/modeling/modeling_category_3_male_presence_all_20260129_193701_hist4s.pkl')
uni_raw_diff_feature_key = 'self.neck_elevation'
uni_raw_diff_feature_color = '#9AC0CD'

In [ ]:
### Plot feature ranking for univariate models (each metric gets its own plot)

plot_feature_ranking(results_file_loc=uni_ranking_results,
                     p_val=uni_ranking_p_val,
                     evaluation_metric='ll',
                     evaluation_metric_name='Negative Log-Likelihood (held-out data)',
                     secondary_metric='score',
                     secondary_metric_name="Accuracy (held-out data)",
                     ignore_features=None,
                     save_plot=False,
                     output_dir=figures_dir)


In [ ]:
### Plot individual filters of univariate models

plot_significant_filters(results_file_loc=uni_filters_results,
                         metric='ll',
                         ignore_features=None,
                         p_val=uni_ranking_p_val,
                         save_plot=False,
                         output_dir=figures_dir)


In [ ]:
### Plot all significant filters in a grid, baseline-corrected, considering univariate models

plot_significant_filters_grid(results_file_loc=uni_filters_grid_results,
                              ignore_features=None,
                              metric='ll',
                              p_val_threshold=uni_ranking_p_val,
                              save_plot=False,
                              output_dir=figures_dir)


In [ ]:
### Plot raw (z-scored) feature differences

plot_raw_feature_difference(pickle_file_path=uni_raw_diff_pkl,
                            feature_key=uni_raw_diff_feature_key,
                            feature_color=uni_raw_diff_feature_color,
                            subset_fraction=0.05,
                            n_bootstraps=100,
                            save_plots=False,
                            output_dir=figures_dir)


**Multinomial target** — predicting a USV's vocal category across *all* categories jointly (a one-vs-rest multinomial classifier) is **not** a scalar target: performance is per-feature **and per-class**, so it does not fit the ranking / filter plotters above and gets its own univariate plotter.

- `plot_univariate_multinomial_performance` — a per-feature × per-class metric heatmap; the `diff_cmap` panel shows each cell's performance relative to the cohort mean. Reads the consolidated univariate-multinomial pickle (`mn_univariate_results`).

In [ ]:
# Parameters -- univariate multinomial visualisation

mn_univariate_results = configure_path('/mnt/scratch/Bartul/modeling/univariate_results/multinomial_categories/univariate_multinomial_categories_male_bin_resize=10_lambda=1E0_l2=1E-1.pkl')

In [ ]:
### Plot performance of univariate multinomial models

plot_univariate_multinomial_performance(
    results_file_loc=mn_univariate_results,
    evaluation_metric='auc',
    evaluation_metric_name='Area Under the ROC Curve',
    secondary_metric='score',
    secondary_metric_name='Balanced Accuracy',
    p_val_threshold=0.05,
    diff_cmap='bwr',
    save_plot=False,
    output_dir=figures_dir)


## 5. Model-selection visualisations

Plots of the forward-stepwise selection trajectory — how held-out performance evolves as features are stacked. As in section 4, **two target families**, separated by the markdown headers below.

**Scalar single-target analyses** — the plotter below serves the forward-stepwise selection of the same three scalar targets (selection is run on the cluster):

- **Vocal onsets** (`VocalOnsetModelingPipeline` → `vocal_onset_model_selection`).
- **Bout parameters** (`BoutParameterPipeline` → `bout_parameter_model_selection`).
- **Binomial USV category** (`VocalCategoryModelingPipeline` → `vocal_category_model_selection`).

One metric line per step (X axis = number of features in the model at that step). Point `msv_results_path` at the chosen analysis's consolidated `model_selection_final_*.pkl`, then run:

- `plot_model_selection_results` — the per-step performance trajectory plus the final accepted model's temporal filters.

In [ ]:
# Parameters -- model-selection visualisations

msv_results_path = configure_path('/mnt/falkner/Bartul/modeling/model_selection_results/male/vocal_onset/model_selection_final_male_intact_partners_onsets_bout_mixed_20260511_203829Z.pkl')

In [ ]:
### Plot model selection results

plot_model_selection_results(
    selection_results_path=msv_results_path,
    metric_secondary='score',
    save_plots=True,
    output_dir=figures_dir,
)


**Multinomial target** — multinomial (one-vs-rest) selection is per-class, so it has its own trajectory / filter / diagnosis plotters, reading the consolidated multinomial selection artifacts.

- `plot_multinomial_selection_trajectory` — primary + secondary metric per forward-stepwise iteration, broken down by class (`mn_trajectory_results`).
- `plot_multinomial_multivariate_filters` — final selected filters (one panel per class × selected feature) with a shared diverging colormap (`mn_filters_results`).
- `plot_multinomial_selection_diagnosis` — how far the multivariate selection departs from picking the top univariate feature per class; base + difference heatmaps (`mn_diagnosis_results`).

In [ ]:
# Parameters -- multinomial model-selection visualisations

mn_trajectory_results = configure_path('/mnt/falkner/Bartul/modeling/model_selection_results/male/multinomial_qlvm_supercategory/model_selection_final_male_intact_partners_multinomial_qlvm_supercategory_mixed_20260515_215415Z.pkl')
mn_filters_results = configure_path('/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male')
mn_diagnosis_results = configure_path('/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner')

In [ ]:
### Plot performance trajectory of model selection for multinomial models

plot_multinomial_selection_trajectory(
    selection_results_path=mn_trajectory_results,
    metric_primary='auc',
    primary_metric_name='Area Under the ROC Curve',
    metric_secondary='score',
    secondary_metric_name="Balanced Accuracy",
    save_plot=False,
    output_dir=figures_dir,
    secondary_ylim_max=0.26)


In [ ]:
### Plot filters of multinomial models (final model selection)

plot_multinomial_multivariate_filters(
    selection_results_path=mn_filters_results,
    history_window_sec=4.0,
    cmap='bwr',
    save_plot=True,
    output_dir=figures_dir,
)


In [ ]:
### Plot how much the final model selection for multinomial models differs from the top-ranked univariate model

plot_multinomial_selection_diagnosis(
    selection_results_path=mn_diagnosis_results,
    cmap_diff='bwr',
    save_plot=True,
    output_dir=figures_dir,
)


## 6. Continuous manifold selection visualisations

These two plotters consume the consolidated artifact written by `continuous_vocal_manifold_model_selection` (forward-stepwise selection for the 2-D acoustic-manifold regression). Same `selection_*.pkl` schema as the multinomial plotters but with continuous regression metrics (`r2_spatial`, `mahalanobis_mae`, `euclidean_mae*`, `pearson_x/y`, `spearman_x/y`) and a 2-D output dim.

- `plot_manifold_selection_trajectory` — two-panel summary: cumulative primary metric across forward-stepwise iterations (left), secondary-metric bars for best-univariate (= the anchor feature) vs the final stacked model (right). Defaults to `r2_spatial` (primary) and `pearson_y` (secondary, the more predictable manifold axis in practice). Higher-is-better metrics grow rightward from chance; error metrics (`euclidean_mae`, etc.) flip the axis and use the step-0 null-model baseline as the chance reference.
- `plot_manifold_multivariate_filters` — final-model per-feature temporal filter atlas. Each subplot is one feature; rows are the two manifold output dims (manifold-x, manifold-y); columns are time bins from `-history_window_sec` to 0. Diverging colormap, symmetric per-feature scaling (`± max|W|`).

In [ ]:
# Parameters -- continuous-manifold visualisations

man_trajectory_results = configure_path('/mnt/falkner/Bartul/modeling/model_selection_results/male/usv_manifold_vae_supercategory/model_selection_final_male_intact_partners_manifold_vae_supercategory_mixed_20260516_155911Z.pkl')
man_filters_results = man_trajectory_results

In [ ]:
### Plot performance trajectory of model selection for the continuous manifold model

plot_manifold_selection_trajectory(
    selection_results_path=man_trajectory_results,
    metric_primary='r2_spatial',
    primary_metric_name='R² (spatial, KDE-weighted)',
    metric_secondary='pearson_y',
    secondary_metric_name='Pearson r (manifold y)',
    save_plot=False,
    output_dir=figures_dir,
)


In [ ]:
### Plot final-model temporal filters of the continuous manifold model

plot_manifold_multivariate_filters(
    selection_results_path=man_filters_results,
    history_window_sec=4.0,
    cmap='RdBu_r',
    save_plot=False,
    output_dir=figures_dir,
)


## 7. CNN deep-learning pipeline (continuous manifold position)

Non-linear baseline for the continuous manifold-position regression. Three cells:

1. **Load** — `runner.load_multivariate_data_blocks(...)` reads the modeling-input pickle from section 1 and stacks the per-feature `(N, T)` matrices into the `(N, F, T)` tensor the 1-D ResNet consumes.
2. **Train** — `runner.run_cnn_training(data_blocks=...)` runs the full training loop and writes `cnn_*_predictions_*.pkl` next to the input pickle.
3. **Visualise** — `DeepResultsVisualizer` exposes five diagnostic modes; pick one via the `choose_analysis` switch (`permutation_test`, `feature_importance`, `spatial_precision_grid`, `error_landscape`, `regional_saliency`).


In [ ]:
# Parameters -- CNN pipeline

cnn_input_pkl = configure_path('/mnt/falkner/Bartul/modeling/modeling_manifold_vae_supercategory_intact_partners_male_20260524_002738.pkl')
cnn_results_pkl = configure_path('/mnt/falkner/Bartul/modeling/cnn_manifold_integrated_predictions_male_QLVM_20260524_205454.pkl')
cnn_choose_analysis = 'regional_saliency'

In [ ]:
### Run CNN to predict vocal manifold positions

# Loads the multivariate feature blocks produced by
# `ContinuousModelingPipeline.extract_and_save_continuous_data()` and
# constructs the 3-D tensor the 1-D ResNet expects. `data_blocks` is then
# handed to `runner.run_cnn_training(...)` in the next cell. The input
# pickle must carry per-USV `supercategory` labels (needed by the saliency
# phase); the pre-flight check inside `run_cnn_training` fails fast if they
# are missing. Source pickle path lives in the Parameters cell (cnn_input_pkl).

runner = NeuralContinuousCNNRunner(modeling_settings=None)

data_blocks = runner.load_multivariate_data_blocks(pkl_path=cnn_input_pkl)


In [ ]:
runner.run_cnn_training(data_blocks=data_blocks)


In [ ]:
### Plot CNN results in predicting vocal manifold positions

# Choose one of the five visualisation modes via `cnn_choose_analysis`
# (Parameters cell). Each mode reads the same `cnn_*_predictions_*.pkl`
# artifact written at the end of `runner.run_cnn_training(...)` and renders
# a different diagnostic projection of the trained network's behaviour.

deep_visualizer = DeepResultsVisualizer(results_pkl_path=cnn_results_pkl,
                                        modeling_settings=None,
                                        visualization_settings=None)

if cnn_choose_analysis == 'permutation_test':
    deep_visualizer.plot_permutation_test(save_plot=False,
                                          output_dir=figures_dir,
                                          file_format='svg')
elif cnn_choose_analysis == 'feature_importance':
    deep_visualizer.plot_feature_importance(snr_threshold=3.0,
                                            error_bar_color='#000000',
                                            save_plot=False,
                                            output_dir=figures_dir,
                                            file_format='svg')
elif cnn_choose_analysis == 'spatial_precision_grid':
    # Torus manifold (e.g. QLVM): pass grid_shape and patch_size.
    # Euclidean manifold (e.g. VAE): leave defaults / pass n_patches=20.
    deep_visualizer.plot_spatial_precision_grid(grid_shape=(4, 4),
                                                patch_size=0.20,
                                                min_samples=25,
                                                plot_type='density',
                                                bg_pt_color='#E0E0E0',
                                                peak_pt_color='#00FFFF',
                                                square_edge_color='#000000',
                                                panel_fontsize=9,
                                                figsize_unit=2.0,
                                                save_plot=False,
                                                output_dir=figures_dir,
                                                file_format='svg')
elif cnn_choose_analysis == 'error_landscape':
    deep_visualizer.plot_error_landscape(gridsize=30,
                                         vmax_percentile=95.0,
                                         save_plot=False,
                                         output_dir=figures_dir,
                                         file_format='svg')
elif cnn_choose_analysis == 'regional_saliency':
    # Region is a circle (centroid + radius) read from the saliency entry.
    # `region_key` must exist in `self.data['saliency_maps']`
    # (e.g. 'supercategory_1' ... 'supercategory_7' for the QLVM pickle).
    deep_visualizer.plot_regional_saliency_inset(
        region_key='supercategory_7',
        category_name='QLVM supercategory 7',
        prediction_plot_type='density',
        radius=0.15,
        smoothing_sigma=10.0,
        save_plot=False,
        output_dir=figures_dir,
        file_format='svg')
else:
    print(f"Option {cnn_choose_analysis} not recognized.")
